# Preprocessament de les dades

Aquest notebook conté el procés de neteja i preparació de les dades utilitzades en el treball.  
A partir del fitxer original `dades_raw.csv`, es construeix una base de dades setmanal agregada per regió, diagnòstic i grup d'edat binari.

El resultat final s'exporta com:

```text
dades_net.csv
```

## Objectiu del preprocessament

El preprocessament té com a objectiu obtenir una base de dades coherent per a l'anàlisi de sèries temporals. En particular:

- es filtren registres sense informació útil de regió o edat;
- es construeix una data setmanal a partir de l'any i la setmana epidemiològica;
- s'agreguen els casos eliminant la dimensió de sexe;
- es recodifiquen els grups d'edat en dues categories: `0-14` i `15+`;
- es calcula la taxa setmanal per cada 100.000 habitants;
- es conserva el període 2012-2024.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


RAW_FILE = Path("dades_raw.csv")
OUTPUT_FILE = Path("dades_net.csv")

START_YEAR = 2012
END_YEAR = 2024

NO_INFO_REGIO = {
    "No disponible",
    "No disponible / desconegut",
    "Desconegut",
}

NO_INFO_EDAT = {
    "No disponible",
    "No especificat",
    "Desconegut",
}

REQUIRED_COLUMNS = {
    "any",
    "setmana_epidemiologica",
    "nom_regio",
    "diagnostic",
    "grup_edat",
    "casos",
    "poblacio",
}

In [2]:
raw = pd.read_csv(RAW_FILE)

missing_columns = REQUIRED_COLUMNS - set(raw.columns)
if missing_columns:
    raise ValueError(f"Falten columnes obligatòries al fitxer original: {missing_columns}")

print(f"Files originals: {len(raw):,}")
print(f"Columnes: {list(raw.columns)}")

raw.head()

Files originals: 1,294,145
Columnes: ['any', 'setmana_epidemiologica', 'nom_regio', 'sexe', 'diagnostic', 'grup_edat', 'casos', 'poblacio']


,any,setmana_epidemiologica,nom_regio,sexe,diagnostic,grup_edat,casos,poblacio
0,2011,40,Alt Pirineu i Aran,Dona,Altres IRA,0,5.0,75.0
1,2011,40,Alt Pirineu i Aran,Dona,Altres IRA,1 i 2,9.0,182.0
2,2011,40,Alt Pirineu i Aran,Dona,Altres IRA,10 a 14,6.0,411.0
3,2011,40,Alt Pirineu i Aran,Dona,Altres IRA,15 a 19,2.0,427.0
4,2011,40,Alt Pirineu i Aran,Dona,Altres IRA,3 i 4,7.0,158.0


## Neteja inicial

Es corregeixen els tipus de les variables temporals i s'eliminen registres sense informació útil de regió o grup d'edat.  
Aquesta decisió evita que les categories no informatives distorsionin les agregacions posteriors.

In [3]:
df = raw.copy()

df["any"] = df["any"].astype(int)
df["setmana_epidemiologica"] = df["setmana_epidemiologica"].astype(int)

df = df.dropna(subset=["nom_regio", "grup_edat"])
df = df[~df["nom_regio"].isin(NO_INFO_REGIO)]
df = df[~df["grup_edat"].isin(NO_INFO_EDAT)]

print(f"Files després de la neteja inicial: {len(df):,}")

Files després de la neteja inicial: 1,222,532


In [4]:
df["date"] = pd.to_datetime(
    df["any"].astype(str)
    + "-W"
    + df["setmana_epidemiologica"].astype(str).str.zfill(2)
    + "-1",
    format="%G-W%V-%u",
)

df[["any", "setmana_epidemiologica", "date"]].head()

,any,setmana_epidemiologica,date
0,2011,40,2011-10-03
1,2011,40,2011-10-03
2,2011,40,2011-10-03
3,2011,40,2011-10-03
4,2011,40,2011-10-03


In [5]:
df_sexe = (
    df.groupby(
        [
            "any",
            "setmana_epidemiologica",
            "date",
            "nom_regio",
            "diagnostic",
            "grup_edat",
        ],
        as_index=False,
    )[["casos", "poblacio"]]
    .sum()
)

print(f"Files després d'agregar per sexe: {len(df_sexe):,}")
df_sexe.head()

Files després d'agregar per sexe: 707,254


,any,setmana_epidemiologica,date,nom_regio,diagnostic,grup_edat,casos,poblacio
0,2011,40,2011-10-03,Alt Pirineu i Aran,Altres IRA,0,12.0,158.0
1,2011,40,2011-10-03,Alt Pirineu i Aran,Altres IRA,1 i 2,18.0,356.0
2,2011,40,2011-10-03,Alt Pirineu i Aran,Altres IRA,10 a 14,7.0,500.0
3,2011,40,2011-10-03,Alt Pirineu i Aran,Altres IRA,15 a 19,4.0,621.0
4,2011,40,2011-10-03,Alt Pirineu i Aran,Altres IRA,20 a 24,1.0,273.0


In [6]:
def edat_binaria(grup_edat: str) -> str:
    '''
    Recodifica els grups d'edat originals en dues categories:
    - 0-14
    - 15+
    '''
    grup_edat = str(grup_edat).strip().lower()

    if grup_edat in {"0", "1 i 2", "3 i 4", "5 a 9", "10 a 14"}:
        return "0-14"

    return "15+"


df_sexe["edat_bin"] = df_sexe["grup_edat"].apply(edat_binaria)

df_sexe[["grup_edat", "edat_bin"]].drop_duplicates().sort_values(
    ["edat_bin", "grup_edat"]
)

,grup_edat,edat_bin
0,0,0-14
1,1 i 2,0-14
2,10 a 14,0-14
6,3 i 4,0-14
11,5 a 9,0-14
3,15 a 19,15+
4,20 a 24,15+
5,25 a 29,15+
7,30 a 34,15+
8,35 a 39,15+


In [7]:
df_edat = (
    df_sexe.groupby(
        [
            "any",
            "setmana_epidemiologica",
            "date",
            "nom_regio",
            "diagnostic",
            "edat_bin",
        ],
        as_index=False,
    )[["casos", "poblacio"]]
    .sum()
)

print(f"Files després d'agregar per edat binària: {len(df_edat):,}")
df_edat.head()

Files després d'agregar per edat binària: 109,215


,any,setmana_epidemiologica,date,nom_regio,diagnostic,edat_bin,casos,poblacio
0,2011,40,2011-10-03,Alt Pirineu i Aran,Altres IRA,0-14,68.0,2245.0
1,2011,40,2011-10-03,Alt Pirineu i Aran,Altres IRA,15+,40.0,8459.0
2,2011,40,2011-10-03,Alt Pirineu i Aran,Bronquiolitis,0-14,5.0,122.0
3,2011,40,2011-10-03,Alt Pirineu i Aran,Escarlatina,0-14,1.0,74.0
4,2011,40,2011-10-03,Alt Pirineu i Aran,Faringoamigdalitis,0-14,23.0,1984.0


In [8]:
df_edat["taxa_setmanal"] = df_edat["casos"] / df_edat["poblacio"] * 100000

df_edat[["casos", "poblacio", "taxa_setmanal"]].head()

,casos,poblacio,taxa_setmanal
0,68.0,2245.0,3028.953229
1,40.0,8459.0,472.869133
2,5.0,122.0,4098.360656
3,1.0,74.0,1351.351351
4,23.0,1984.0,1159.274194


In [9]:
df_final = df_edat[
    (df_edat["any"] >= START_YEAR)
    & (df_edat["any"] <= END_YEAR)
].copy()

print(f"Període final: {df_final['any'].min()}-{df_final['any'].max()}")
print(f"Files finals: {len(df_final):,}")

df_final.head()

Període final: 2012-2024
Files finals: 101,010


,any,setmana_epidemiologica,date,nom_regio,diagnostic,edat_bin,casos,poblacio,taxa_setmanal
1810,2012,1,2012-01-02,Alt Pirineu i Aran,Altres IRA,0-14,44.0,1773.0,2481.669487
1811,2012,1,2012-01-02,Alt Pirineu i Aran,Altres IRA,15+,55.0,8731.0,629.939297
1812,2012,1,2012-01-02,Alt Pirineu i Aran,Bronquiolitis,0-14,9.0,401.0,2244.389027
1813,2012,1,2012-01-02,Alt Pirineu i Aran,Faringoamigdalitis,0-14,12.0,853.0,1406.799531
1814,2012,1,2012-01-02,Alt Pirineu i Aran,Faringoamigdalitis,15+,25.0,5586.0,447.547440


## Comprovacions finals

In [10]:
checks = {
    "any_in_columns": "any" in df_final.columns,
    "min_year": df_final["any"].min(),
    "max_year": df_final["any"].max(),
    "min_week": df_final["setmana_epidemiologica"].min(),
    "max_week": df_final["setmana_epidemiologica"].max(),
    "missing_region": df_final["nom_regio"].isna().sum(),
    "missing_age_bin": df_final["edat_bin"].isna().sum(),
    "missing_rate": df_final["taxa_setmanal"].isna().sum(),
}

pd.Series(checks, name="valor")

any_in_columns     True
min_year           2012
max_year           2024
min_week              1
max_week             53
missing_region        0
missing_age_bin       0
missing_rate          0
Name: valor, dtype: object

In [11]:
df_final.to_csv(OUTPUT_FILE, index=False)

print(f"Fitxer exportat correctament: {OUTPUT_FILE}")

Fitxer exportat correctament: dades_net.csv
